In [1]:
import matplotlib.pyplot as plt
import pickle

from pmbrl.model2 import Model
from pmbrl.data import Experiment_Data, get_data_expanded

In [2]:
# nome_do_arquivo = 'regular_normilized.pkl'
nome_do_arquivo = 'regular.pkl'

with open(nome_do_arquivo, 'rb') as arquivo:
    exp = pickle.load(arquivo)
    data = exp['data']
    model = exp['model']

del exp
del arquivo

# Calculating AVG

In [3]:
expansions = {
    'estimated_p': ['p0', 'p1'],
    's_': ['s0', 's1', 's2', 's3'],
    's__': ['s_0', 's_1', 's_2', 's_3'],
}

df = get_data_expanded(data.evaluation_data, expansions)
df_avgs = df.groupby('episode')[['p0', 'p1']].mean().reset_index()

def get_avg_per_epi(row):
    return df_avgs.loc[df_avgs['episode'] == row.episode][['p0', 'p1']].values[0]
df[['avg_p0', 'avg_p1']] = df.apply(get_avg_per_epi, axis=1, result_type='expand')

df[['episode', 'avg_p0', 'avg_p1']]

,episode,avg_p0,avg_p1
0,0,0.289667,-0.731667
1,0,0.289667,-0.731667
2,0,0.289667,-0.731667
3,0,0.289667,-0.731667
4,0,0.289667,-0.731667
...,...,...,...
2122,99,0.783933,-0.438133
2123,99,0.783933,-0.438133
2124,99,0.783933,-0.438133
2125,99,0.783933,-0.438133


In [4]:
df_std = df.groupby('episode')[['p0', 'p1']].std().reset_index()
df_min = df.groupby('episode')[['p0', 'p1']].min().reset_index()
df_max = df.groupby('episode')[['p0', 'p1']].max().reset_index()
df_count = df.groupby('episode')[['p0', 'p1']].count().reset_index()
df_stats = df_avgs[['episode']].copy()

df_stats[['avg_p0', 'avg_p1']] = df_avgs[['p0', 'p1']]
df_stats[['std_p0', 'std_p1']] = df_std[['p0', 'p1']]
df_stats[['min_p0', 'min_p1']] = df_min[['p0', 'p1']]
df_stats[['max_p0', 'max_p1']] = df_max[['p0', 'p1']]
df_stats['range_p0'] = df_stats['max_p0'] - df_stats['min_p0']
df_stats['range_p1'] = df_stats['max_p1'] - df_stats['min_p1']
df_stats[['count']] = df_count[['p0']]

df_stats

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
0,0,0.289667,-0.731667,0.098879,0.053433,0.140,-0.788,0.398,-0.660,0.258,0.128,6
1,1,1.138000,-0.025556,0.113548,0.136474,0.937,-0.209,1.290,0.192,0.353,0.401,9
2,2,0.823625,-0.373188,0.084375,0.072406,0.679,-0.508,1.015,-0.234,0.336,0.274,16
3,3,0.841421,-0.347842,0.054531,0.079759,0.758,-0.516,0.982,-0.185,0.224,0.331,19
4,4,1.222615,-0.097077,0.112848,0.172158,1.042,-0.289,1.416,0.220,0.374,0.509,13
...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.577467,-0.560200,0.235243,0.055751,0.331,-0.655,1.122,-0.410,0.791,0.245,15
96,96,1.006658,-0.176342,0.103198,0.109904,0.830,-0.365,1.181,0.021,0.351,0.386,38
97,97,0.852743,-0.382314,0.179858,0.188995,0.607,-0.888,1.180,-0.110,0.573,0.778,35
98,98,0.838867,-0.279795,0.112689,0.129926,0.612,-0.599,1.180,0.001,0.568,0.600,83


In [5]:
df_stats.describe()

,episode,avg_p0,avg_p1,std_p0,std_p1,min_p0,min_p1,max_p0,max_p1,range_p0,range_p1,count
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000
mean,49.500000,0.906208,-0.317437,0.143779,0.130188,0.691670,-0.523910,1.154170,-0.085080,0.462500,0.438830,21.270000
std,29.011492,0.221704,0.195943,0.091574,0.061860,0.239077,0.194007,0.233155,0.263192,0.273311,0.187429,14.489882
min,0.000000,0.289667,-1.051143,0.020109,0.028146,-0.099000,-1.238000,0.398000,-0.884000,0.080000,0.076000,3.000000
25%,24.750000,0.786711,-0.441042,0.080908,0.082783,0.547500,-0.620000,1.010000,-0.266500,0.270750,0.323750,11.000000
50%,49.500000,0.914640,-0.327943,0.117502,0.123515,0.726000,-0.515500,1.171500,-0.073500,0.405000,0.424000,16.500000
75%,74.250000,1.053416,-0.153926,0.186259,0.162805,0.830250,-0.387500,1.293000,0.132500,0.600250,0.521250,29.250000
max,99.000000,1.420000,0.043091,0.435407,0.326090,1.155000,-0.179000,1.641000,0.456000,1.248000,1.042000,83.000000


# Infering with avg

In [6]:
import torch
import torch.nn as nn

m = model.transition_estimator.state_layer

input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'avg_p0', 'avg_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0385)

In [7]:
results = data.get_evaluation_metrics()
results.rse.mean()

np.float64(0.046223789374706156)

# Optimazing p

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim

history = []

for epi, p1, p2 in df_avgs.values:
    print(f'{epi:}')
    d = df[df['episode'] == epi].copy().reset_index(drop=True)

    target_value = torch.tensor(d[['s_0', 's_1', 's_2', 's_3']].values)
    param = torch.tensor([p1, p2], requires_grad=True)
    print(f'initial value for input Param: {[round(p,4) for p in param.tolist()]}') 
    input_values = torch.tensor(d[['s0', 's1', 's2', 's3', 'a_']].values) 

    learning_rate = 0.01
    num_epochs = 500

    optimizer = optim.Adam([param], lr=learning_rate)
    criterion = nn.MSELoss(reduction='none')

    for p in m.parameters():
        p.requires_grad = False

    for epoch in range(num_epochs):
        # Forward pass
        state_inputs = torch.concat([input_values, param.repeat((input_values.shape[0], 1))], dim=1)
        output = m(state_inputs.float())  # Add batch dimension

        # Calculate the loss
        # loss = criterion(output.float(), target_value.float())
        
        def normilize(v): 
            mins = input_values[:,:-1].min(axis=0).values.repeat((v.shape[0], 1))
            maxs = input_values[:,:-1].max(axis=0).values.repeat((v.shape[0], 1))
            rang = maxs - mins
            return (v - mins) / rang
        open_loss = criterion(normilize(output).float(), normilize(target_value).float())
        # open_loss = criterion(output.float(), target_value.float())
        loss = torch.sqrt(open_loss.sum(axis=1).mean())
        # loss = torch.sqrt(open_loss.mean(axis=1).sum())
        # loss = open_loss.mean()
        # loss = open_loss.sum()

        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # history.append((epoch, param.tolist(), loss.item()))

        # if (epoch + 1) % 100 == 0:
        #     # print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {param.item():.4f}, Output: {output.item():.4f}')
        #     print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')
    history.append({'episode': epi, 'optimized_p': param.tolist(), 'loss':  loss.item()})
    print(f'Loss: {loss.item():.4f}, Input Param: {[round(p,4) for p in param.tolist()]}')


0.0
initial value for input Param: [0.2897, -0.7317]
Loss: 0.2035, Input Param: [0.2428, -0.8296]
1.0
initial value for input Param: [1.138, -0.0256]
Loss: 0.1907, Input Param: [1.087, -0.0349]
2.0
initial value for input Param: [0.8236, -0.3732]
Loss: 0.1875, Input Param: [0.7677, -0.3538]
3.0
initial value for input Param: [0.8414, -0.3478]
Loss: 0.1366, Input Param: [0.7699, -0.3282]
4.0
initial value for input Param: [1.2226, -0.0971]
Loss: 0.1825, Input Param: [1.1801, -0.0766]
5.0
initial value for input Param: [1.074, -0.0763]
Loss: 0.2201, Input Param: [1.0301, -0.0488]
6.0
initial value for input Param: [1.0125, -0.1002]
Loss: 0.1990, Input Param: [0.9676, -0.09]
7.0
initial value for input Param: [1.2915, -0.0644]
Loss: 0.1777, Input Param: [1.2279, -0.0629]
8.0
initial value for input Param: [0.4906, -0.6317]
Loss: 0.3805, Input Param: [0.4854, -0.7273]
9.0
initial value for input Param: [0.9861, -0.2483]
Loss: 0.1483, Input Param: [0.9333, -0.2402]
10.0
initial value for in

In [9]:
import pandas as pd
df_optim = pd.DataFrame(history)


expansions = {
    'optimized_p': ['opt_p0', 'opt_p1'],
}

df_optim = get_data_expanded(df_optim, expansions)[['episode', 'loss', 'opt_p0', 'opt_p1']]
df_optim.loss.mean()

np.float64(0.2010582120716572)

In [10]:
import torch
import torch.nn as nn

def get_opt_per_epi(row):
    return df_optim.loc[df_optim['episode'] == row.episode][['opt_p0', 'opt_p1']].values[0]
df[['opt_p0', 'opt_p1']] = df.apply(get_opt_per_epi, axis=1, result_type='expand')


input_values = torch.tensor(df[['s0', 's1', 's2', 's3', 'a_', 'opt_p0', 'opt_p1']].values) 
target_value = torch.tensor(df[['s_0', 's_1', 's_2', 's_3']].values)

for p in m.parameters():
    p.requires_grad = False

output = m(input_values.float()) 


def normilize(v): 
        mins = input_values[:,0:4].min(axis=0).values.repeat((v.shape[0], 1))
        maxs = input_values[:,0:4].max(axis=0).values.repeat((v.shape[0], 1))
        rang = maxs - mins
        return (v - mins) / rang
loss = nn.MSELoss(reduction='none')(normilize(output).float(), normilize(target_value).float())
# loss = nn.MSELoss(reduction='none')(normilize(output).float(), (target_value).float())
loss = torch.sqrt(loss.mean(axis=0).sum())
loss

tensor(0.0396)

In [11]:
del model
del data